# 03 — Cost-Based Evaluation

Assumptions:
- False negative cost = the transaction Amount (money lost to fraud)
- False positive cost = $5 (analyst review + customer friction; this is an assumption)
- True positive and true negative cost = $0

These are stated assumptions, not ground truth. The sensitivity analysis shows how conclusions
change as the FP cost varies from $1 to $50.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fraud.data import RES, FIG

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

cost_res = pd.read_csv(RES / "cost_results.csv")
sens = pd.read_csv(RES / "cost_sensitivity.csv")

print("=== Cost results by model and threshold rule ===")
print(cost_res.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pivot = cost_res.pivot_table(index="model", columns="rule", values="cost")
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("total cost (USD)")
ax.set_title("Test-set cost by model and threshold rule\n(lower is better)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG / "cost_by_model.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("=== Sensitivity: total cost by FP cost assumption ===")
print(sens.pivot(index="fp_cost", columns="model", values="cost"))

print("\n=== XGB threshold and alert count vs FP cost ===")
print(sens[sens.model == "xgb"][["fp_cost", "threshold", "alerts", "caught"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for model, grp in sens.groupby("model"):
    axes[0].plot(grp.fp_cost, grp.cost, marker="o", label=model)
    if model == "xgb":
        axes[1].plot(grp.fp_cost, grp.threshold, marker="o", color="#D65F5F")

axes[0].set_xlabel("FP cost assumption (USD)")
axes[0].set_ylabel("total cost (USD)")
axes[0].set_title("Total cost sensitivity to FP assumption")
axes[0].legend()
axes[1].set_xlabel("FP cost assumption (USD)")
axes[1].set_ylabel("optimal threshold")
axes[1].set_title("XGBoost: optimal threshold vs FP cost")
fig.tight_layout()
fig.savefig(FIG / "cost_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import json
from fraud.cost import total_cost, flag_nothing_cost

d = np.load(RES / "scores.npz")
yte, ate = d["yte"], d["amt_te"]

# amount-aware rule: flag when p * Amount > FP_cost (i.e., p > FP_cost / Amount)
# requires calibrated probabilities
cal_te = np.load(RES / "xgb_te_calibrated.npy")
FP = 5.0
per_tx_threshold = FP / (ate + 1e-6)
amount_aware_pred = cal_te > per_tx_threshold

cfg = json.load(open(RES / "chosen_threshold.json"))
global_pred = d["xgb_te"] >= cfg["threshold"]

c_global = total_cost(yte, global_pred, ate, FP)
c_aware = total_cost(yte, amount_aware_pred, ate, FP)
c_none = flag_nothing_cost(yte, ate)

print(f"Flag nothing:          ${c_none:.2f}")
print(f"Global threshold:      ${c_global:.2f}  (alerts: {global_pred.sum()})")
print(f"Amount-aware rule:     ${c_aware:.2f}  (alerts: {amount_aware_pred.sum()})")

## Key findings

The cost results above give the actual numbers from this run. Points to highlight:

- The cost-optimal threshold is typically higher than 0.5, because false positives accumulate
  quickly at lower thresholds.
- As the FP cost assumption rises from $1 to $50, the optimal threshold increases and fewer
  transactions are flagged. At very high FP costs, the model may recommend flagging almost nothing.
- The amount-aware rule (flag when p × Amount > FP cost) is theoretically optimal for
  per-transaction decisions under these cost assumptions.